# 1-state energy scan across momenta
This notebook plots all available momenta on one energy-scan axis.
Edit the configuration cell to change the source tables, colors, or save path.


In [1]:
from pathlib import Path
import math
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path(r"/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit")
STATE = 1
NS = 64
LATTICE_SPACING_FM = 0.076
RESULTS_ROOT = ROOT / "results_nstate_fit_1state"
OUTPUT_PDF = ROOT / "nstate_energy_scan_l64c64a076_all_pz_1state.pdf"

# Edit this block to change the source tables, axis styling, or save path.
SHOW_DISPERSION = True
FIGSIZE = (5, 3.2)
LEGEND_NCOL = 1
FONT_FAMILY = 'Times New Roman'
MATH_FONT = 'custom'
MATH_TEXT = {
    'mathtext.fontset': MATH_FONT,
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
}
X_SHIFT = 0.08
YLIM = (0, 3)


In [2]:

HBAR_C_MEV_FM = 197.3269804


def lattice_energy_to_gev(values: np.ndarray) -> np.ndarray:
    return np.asarray(values, dtype=float) * (HBAR_C_MEV_FM / LATTICE_SPACING_FM) / 1000.0


def parse_pz_from_dataset(dataset_dir: Path) -> int:
    match = re.search(r'pz(\d+)', dataset_dir.name)
    if match is None:
        raise ValueError(f'could not infer pz from {dataset_dir.name}')
    return int(match.group(1))


def momentum_to_gev(pz: int) -> float:
    return (2.0 * np.pi * pz / (NS * LATTICE_SPACING_FM)) * (HBAR_C_MEV_FM / 1000.0)


def parse_target_energy(summary_path: Path, pz: int) -> float:
    target_key = f'1state_target_energy_pz{pz}'
    with summary_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                continue
            tokens = stripped.split()
            if len(tokens) >= 2 and tokens[0] == target_key:
                value = float(tokens[1])
                if not math.isfinite(value):
                    raise ValueError(f'non-finite target energy in {summary_path}')
                return value
    raise ValueError(f'missing {target_key} in {summary_path}')


def load_series():
    series = []
    for table_path in sorted(RESULTS_ROOT.glob(f'*/tables/*_symmetric_{STATE}state_tmax*_fits.txt')):
        table = np.atleast_2d(np.loadtxt(table_path, dtype=float))
        dataset_dir = table_path.parent.parent
        pz = parse_pz_from_dataset(dataset_dir)
        chi2_column = 7
        selected = table[np.nanargmin(table[:, chi2_column])]
        nstates = STATE
        amp_mean_start = 9
        energy_mean_start = amp_mean_start + 2 * nstates
        energy_err_start = energy_mean_start + nstates
        tmins = table[:, 0].astype(int)
        energies = lattice_energy_to_gev(table[:, energy_mean_start + (nstates - 1)])
        errors = lattice_energy_to_gev(table[:, energy_err_start + (nstates - 1)])
        target_gev = None
        if SHOW_DISPERSION:
            summary_path = dataset_dir / f'{dataset_dir.name}_symmetric_summary.txt'
            target_gev = lattice_energy_to_gev(np.array([parse_target_energy(summary_path, pz)]))[0]
        series.append({
            'pz': pz,
            'tmins': tmins,
            'energies': energies,
            'errors': errors,
            'target_gev': target_gev,
        })
    series.sort(key=lambda item: item['pz'])
    return series



def plot_combined_energy_scan():
    series = load_series()
    plt.rcParams.update({'font.family': FONT_FAMILY, **MATH_TEXT})
    fig, ax = plt.subplots(figsize=FIGSIZE)
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(series), 1)))
    xmins = []
    xmaxs = []
    nseries = len(series)
    for idx, item in enumerate(series):
        color = colors[idx % len(colors)]
        x = item['tmins']
        x_shift = (idx - (nseries - 1) / 2.0) * X_SHIFT
        x_plot = x + x_shift
        y = item['energies']
        yerr = item['errors']
        xmins.append(float(np.min(x_plot)))
        xmaxs.append(float(np.max(x_plot)))
        momentum_gev = momentum_to_gev(item['pz'])
        label = rf'$P_z={momentum_gev:.2f}$ [GeV]'
        ax.errorbar(x_plot, y, yerr=yerr, fmt='o', ms=4, color=color, label=label)
        if SHOW_DISPERSION and item['target_gev'] is not None:
            ax.hlines(
                item['target_gev'],
                xmin=float(np.min(x_plot)),
                xmax=float(np.max(x_plot)),
                colors=color,
                linestyles='--',
                linewidth=1.2,
                alpha=0.9,
            )
    ax.set_ylim(*YLIM)
    ax.set_xlim(min(xmins), max(xmaxs))
    ax.set_xlabel(r'$t_{\rm min}/a$')
    ax.set_ylabel(r'$E_0$ [GeV]')
    ax.grid(False)
    ax.tick_params(direction='in', top=True, right=True)
    ax.legend(ncol=LEGEND_NCOL, fontsize=8, frameon=True)
    fig.tight_layout()
    fig.savefig(OUTPUT_PDF, bbox_inches='tight')
    return fig


fig = plot_combined_energy_scan()
plt.show()
print(f'Saved {OUTPUT_PDF}')


Saved /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/nstate_energy_scan_l64c64a076_all_pz_1state.pdf


/var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/ipykernel_75637/2978764476.py:109: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
fig = plot_combined_energy_scan()
plt.show()
fig


/var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/ipykernel_75637/1922144824.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 500x320 with 1 Axes>